In [4]:
from bs4 import BeautifulSoup
import re
import tiktoken

# Load HTML (normally from a file or variable; here you paste your string)
with open("index.html", "r", encoding="utf-8") as f:
    html_content = f.read()

soup = BeautifulSoup(html_content, "html.parser")
articles = soup.find_all("li", class_="article-item")

enc = tiktoken.encoding_for_model("gpt-3.5-turbo")

title_token_count = 0
title_word_count = 0
abstract_token_count = 0
abstract_word_count = 0

for art in articles:
    title = art.get("data-title", "").strip()
    abstract = art.get("data-abstract", "").strip()

    # Count words
    title_word_count += len(title.split())
    abstract_word_count += len(abstract.split())

    # Count tokens
    title_token_count += len(enc.encode(title))
    abstract_token_count += len(enc.encode(abstract))

summary = {
    "Total Articles": len(articles),
    "Title Word Count": title_word_count,
    "Title Token Count": title_token_count,
    "Abstract Word Count": abstract_word_count,
    "Abstract Token Count": abstract_token_count,
}

summary


{'Total Articles': 7424,
 'Title Word Count': 93924,
 'Title Token Count': 144468,
 'Abstract Word Count': 634392,
 'Abstract Token Count': 915946}

In [16]:
import tiktoken
enc = tiktoken.encoding_for_model("gpt-3.5-turbo")
text = 'ciao come va ttapposht? mah insomma mezzi alla bang annd mezz al posht'
print(f"tokens: {len(enc.encode(text))}, chars: {len(text)}")

tokens: 24, chars: 71


In [6]:
import random
from bs4 import BeautifulSoup

def extract_random_articles_from_html(html_path, n_articles=10):
    with open(html_path, 'r', encoding='utf-8') as f:
        soup = BeautifulSoup(f, 'html.parser')

    # Find all journals
    journal_sections = soup.find_all("div", class_="accordion-item")

    all_articles = []

    for section in journal_sections:
        journal_name = section.find("h3", class_="journal-header").text.strip().replace("–", "-").replace("—", "-")
        articles = section.find_all("li", class_="article-item")
        for article in articles:
            title = article.get("data-title", "").strip()
            abstract = article.get("data-abstract", "").strip()
            authors = article.get("data-authors", "").strip()
            doi_tag = article.find("a", class_="read-more-link")
            doi = doi_tag['href'] if doi_tag else "N/A"

            all_articles.append({
                "journal": journal_name,
                "title": title,
                "abstract": abstract,
                "authors": authors,
                "doi": doi
            })

    # Shuffle and pick random articles from distinct journals
    random.shuffle(all_articles)
    selected = {}
    for art in all_articles:
        if len(selected) >= n_articles:
            break
        journal = art['journal']
        if journal not in selected:
            selected[journal] = art

    return list(selected.values())

# Example usage
if __name__ == "__main__":
    selected_articles = extract_random_articles_from_html("index.html", n_articles=10)
    for i, art in enumerate(selected_articles, 1):
        #print(f"\nArticle {i}:")
        #print(f"Journal:  {art['journal']}")
        print(f"Article {i} - Title:    {art['title']}")
        #print(f"Authors:  {art['authors']}")
        #print(f"DOI:      {art['doi']}")
        #print(f"Abstract: {art['abstract'][:300]}...")


Article 1 - Title:    attentional state, not trait, predicts test performance in video-based learning
Article 2 - Title:    detecting suicide risk in bipolar disorder patients from lymphoblastoid cell lines genetic signatures
Article 3 - Title:    glial fibrillary acidic protein vs. s100b to identify astrocytes impacted by sex and high fat diet
Article 4 - Title:    a novel ipsc model of bryant-li-bhoj neurodevelopmental/neurodegenerative syndrome demonstrates the role of histone h3.3 in chromatin dynamics, neuronal differentiation, and maturation
Article 5 - Title:    musical rhythm abilities and risk for developmental speech-language problems and disorders: epidemiological and polygenic associations
Article 6 - Title:    crispr activation for scn2a-related neurodevelopmental disorders
Article 7 - Title:    pb2 and np of north american h5n1 virus drive immune cell replication and systemic infections
Article 8 - Title:    neurodegenerative features of damage to deiters neurons in the l

In [1]:
from falcon_tests import ArticleClassifier2


In [2]:
ac = ArticleClassifier2()

In [4]:
ac.classify('oxytocin attenuates the retrieval of methamphetamine-associated reward memories by enhancing adult hippocampal neurogenesis in mice',parse=True)

Prompt length in chars: 771 and tokens: 173


{'neuroscience': 'yes',
 'type': 'article',
 'generic_keywords': ['oxytocin',
  'reward memory',
  'neurogenesis',
  'methamphetamine',
  'mice'],
 'specific_keywords': ['oxytocin',
  'reward',
  'neurogenesis',
  'methamphetamine',
  'behavioral']}

In [5]:
out = ac.classify_batch([('the glutaminase inhibitor jhu-083 mitigates cognitive dysfunction in a mouse model of post-traumatic stress disorder',''),
                   ('development of coherent cortical responses reflects increased discriminability of feedforward inputs and their alignment with recurrent circuits',''),
                         ("cancer-associated fibroblasts shape the formation of budding cancer cells at the invasive front of human colorectal cancer","")
                         ],parse=True)
out

Prompt length in chars: 1213 and tokens: 261


[{'neuroscience': 'yes',
  'type': 'article',
  'generic_keywords': ['cognitive function',
   'stress',
   'mice',
   'model',
   'behavior'],
  'specific_keywords': ['glutaminase',
   'JHU-083',
   'PTSD',
   'cognition',
   'dysfunction']},
 {'neuroscience': 'yes',
  'type': 'article',
  'generic_keywords': ['neural development', 'brain activity', 'learning'],
  'specific_keywords': ['cortical responses',
   'feedforward',
   'recurrent circuits']},
 {'neuroscience': 'yes',
  'type': 'article',
  'generic_keywords': ['cancer formation',
   'human tissue',
   'cellular processes'],
  'specific_keywords': ['fibroblasts', 'colorectal cancer', 'invasive front']}]

In [6]:
ac.known_keywords

{'PTSD',
 'adult hippocampus',
 'alignment with recurrent circuits',
 'animal model',
 'brain development',
 'brain function',
 'cognitive deficits',
 'cognitive dysfunction',
 'cortical maturation',
 'cortical responses',
 'discriminability',
 'drug effects',
 'feedforward inputs',
 'feedforward pathways',
 'glutaminase inhibitor',
 'memory',
 'methamphetamine',
 'mitigation',
 'mouse model',
 'neurogenesis',
 'oxytocin',
 'post-traumatic stress disorder',
 'retrieval',
 'reward memories',
 'treatment'}

In [20]:
!ollama serve

Error: listen tcp 127.0.0.1:11434: bind: address already in use


In [23]:
!ps aux | grep ollama

morning+   61703  0.0  0.0 1930512 31880 pts/2   Tl   22:34   0:00 ollama run mistral
ollama     64555  0.8  0.0 2004240 32164 ?       Ssl  22:51   0:00 /usr/local/bin/ollama serve
morning+   64581 83.3  0.0 231928  3720 pts/3    Ss+  22:51   0:00 /bin/bash -c ps aux | grep ollama
morning+   64583  0.0  0.0 231252  2392 pts/3    S+   22:51   0:00 grep ollama


In [18]:
!sudo pkill ollama

[sudo] password for morningrise: 
